# 03. Demo Seminar 2: phát hiện package PyPI độc bằng LLM, RAG, KG-RAG

**Câu hỏi.** Khi phân loại một package PyPI là độc hay lành, truy hồi tri thức (RAG), truy hồi lai (hybrid), đồ thị hành vi (KG-RAG)
và bước xác minh bằng chứng có giúp gì so với LLM thuần? Paper tham khảo (*Detecting Malicious Source Code in PyPI Packages with LLMs:
Does RAG Come in Handy?*) thấy RAG đơn giản giúp ít, fine-tune giúp nhiều. Ở đây ta tách nhỏ câu hỏi theo plan: RQ1 (truy hồi), RQ2 (hybrid),
RQ3 (cấu trúc đồ thị), RQ4 (xác minh), trên cùng sample id, cùng reader, cùng ngân sách token.

Notebook chỉ **đọc kết quả đã lưu** (notebook 01, 02), không gọi LLM, không cần GPU. Có `results/snapshot/` thì đọc từ đó, nếu không thì đọc trên Drive.

| Chỉ số | Ý nghĩa |
|---|---|
| F1 malicious (strict) | lớp dương là *độc*; câu abstain tính là sai |
| FPR | tỉ lệ package lành bị gọi là độc |
| coverage | tỉ lệ package có câu trả lời (không abstain) |
| unsupported claim rate | tỉ lệ claim không được fact tĩnh + item được trích hỗ trợ |
| citation accuracy | tỉ lệ trích dẫn trỏ tới item của package thật sự cho thấy hành vi |

In [ ]:
# ==== Setup: chỉ đọc kết quả ====
import os, sys, json, glob, shutil
import numpy as np, pandas as pd
try:
    from google.colab import drive; drive.mount('/content/drive'); IN_COLAB = True
except ImportError:
    IN_COLAB = False
DRIVE_BASE = '/content/drive/MyDrive/ThS/Research/colab' if IN_COLAB else os.environ.get('PYPI_RAG_BASE', os.path.abspath('../..'))   # SỬA NẾU KHÁC
CODE_DIR = f'{DRIVE_BASE}/pypi_malware_rag'
WORK = os.environ.get('PYPI_RAG_WORK', f'{DRIVE_BASE}/pypi_rag_work')
SNAPSHOT = f'{CODE_DIR}/results/snapshot'
sys.path.insert(0, CODE_DIR)
pd.set_option('display.max_colwidth', 120)

In [ ]:
# ==== (Chỉ một lần, trên Colab) snapshot gọn: results.json, report.md, bảng lỗi, ca minh hoạ ====
# Sau đó tải folder pypi_malware_rag về máy: notebook này chạy được ở bất kỳ đâu.
if not os.path.exists(f'{SNAPSHOT}/MANIFEST.json') and os.path.isdir(f'{WORK}/results'):
    os.makedirs(SNAPSHOT, exist_ok=True); copied = []
    for d in sorted(glob.glob(f'{WORK}/results/r0*')):
        if not os.path.exists(f'{d}/results.json'): continue
        dst = f'{SNAPSHOT}/{os.path.basename(d)}'; os.makedirs(dst, exist_ok=True)
        for fn in ('results.json', 'report.md', 'error_groups.csv', 'failure_cases.json', 'reproduction.json'):
            if os.path.exists(f'{d}/{fn}'): shutil.copy(f'{d}/{fn}', dst)
        copied.append(os.path.basename(d))
    for fn in ('FROZEN.json', 'upstream_audit.json', 'results/cases.json'):
        if os.path.exists(f'{WORK}/{fn}'): shutil.copy(f'{WORK}/{fn}', SNAPSHOT)
    json.dump({'runs': copied}, open(f'{SNAPSHOT}/MANIFEST.json', 'w'), indent=1)
RES = SNAPSHOT if os.path.exists(f'{SNAPSHOT}/MANIFEST.json') else f'{WORK}/results'
def _j(p):
    try: return json.load(open(p, encoding='utf-8'))
    except (FileNotFoundError, json.JSONDecodeError): return None
RUNS = {os.path.basename(d): _j(f'{d}/results.json') for d in sorted(glob.glob(f'{RES}/r0*')) if _j(f'{d}/results.json')}
print('nguồn:', RES, '|', len(RUNS), 'run:', ', '.join(RUNS))
NAMES = {'e0': 'E0 LLM thuần', 'e1': 'E1 Simple RAG', 'e2': 'E2 Hybrid RAG', 'e2_bm25': 'E2 chỉ BM25', 'e2_rerank': 'E2 + rerank',
         'e3': 'E3 KG-RAG', 'e3a': 'E3a cùng fact dạng câu', 'e4': 'E4 KG-RAG + xác minh', 'e4_filter': 'E4 chỉ lọc claim',
         'e4_cite': 'E4 chỉ kiểm trích dẫn', 'e0_malllm': 'mal-LLM zero-shot', 'e1_malllm_yara': 'mal-LLM RAG-YARA',
         'e1_malllm_code': 'mal-LLM RAG-malcode'}
COLS = {'strict/f1_malicious': 'F1 độc', 'strict/macro_f1': 'macro F1', 'strict/fpr': 'FPR', 'strict/fnr': 'FNR', 'coverage': 'coverage',
        'evidence/unsupported_claim_rate': 'claim không chứng cứ', 'evidence/citation_accuracy': 'trích dẫn đúng', 'tokens_in_mean': 'token vào'}
def table(run, methods, cols=COLS):
    R = RUNS.get(run)
    if not R: print('(chưa có', run, ')'); return None
    df = pd.DataFrame({NAMES.get(m, m): {v: R['summaries'][m].get(k) for k, v in cols.items()} for m in methods if m in R['summaries']}).T
    display(df.astype(float).round(3)); return df
def cmp(run, pairs):
    R = RUNS.get(run)
    if not R: return
    rows = []
    for k in pairs:
        c = R['comparisons'].get(k)
        if c:
            b = c['f1_malicious_strict']
            rows.append({'so sánh': k, 'câu hỏi': c.get('question'), 'n': c['n'], 'dF1': round(b['delta'], 3),
                         'CI 95%': f"[{b['ci_low']:+.3f}, {b['ci_high']:+.3f}]", 'p': round(b['p_value'], 3),
                         'McNemar p': round(c['mcnemar']['p_value'], 3)})
    if rows: display(pd.DataFrame(rows))
MAIN = [r for r in RUNS if r.startswith(('r03_test', 'r04_test')) and not r.endswith('_nodup')]
SYN = [r for r in RUNS if r.startswith('r05_test')]

## Phần 0. Dữ liệu

In [ ]:
frz = _j(f'{RES}/FROZEN.json') or _j(f'{WORK}/FROZEN.json')
if frz: display(pd.DataFrame({n: {sp: v['n'] for sp, v in d.items()} for n, d in frz['datasets'].items()}).fillna(0).astype(int))

## Phần 1. LLM thuần (E0) và tái lập mal-LLM

**1. Hoạt động thế nào.** LLM đọc trực tiếp biểu diễn của package (metadata + các đoạn mã quanh API nhạy cảm, có số dòng) và trả về verdict JSON
kèm claim trích dẫn đoạn mã. Không có tri thức ngoài.

**2. Ưu / nhược.** Rẻ, nhanh, không cần hạ tầng truy hồi. Nhưng model phải tự biết mẫu hành vi độc; với API nhạy cảm dùng hợp pháp (HTTP client, subprocess
cố định) dễ báo nhầm, và lời giải thích có thể nêu hành vi không có trong mã.

**3. Hợp với dữ liệu nào.** Khi tín hiệu nằm ngay trong đầu vào và rõ ràng (một dòng `exec(base64.b64decode(...))` trong setup.py).

In [ ]:
for r in [r for r in RUNS if r.startswith('r01_')]:
    rep = _j(f'{RES}/{r}/reproduction.json')
    print('#####', r); display(pd.DataFrame(rep).set_index('method')) if rep else table(r, ['e0_malllm', 'e1_malllm_yara', 'e1_malllm_code'])
for r in MAIN:
    print('#####', r); table(r, ['e0', 'e1', 'e2', 'e3', 'e4'])

## Phần 2. RAG và Hybrid RAG (E1, E2)

**1. Hoạt động thế nào.** Truy vấn = các đoạn mã đáng ngờ + danh sách API/hành vi. E1 lấy top-5 tài liệu KB theo embedding (luật, pattern,
ví dụ mã độc/lành của **train**, PyGuard, YARA). E2 lấy song song theo embedding và BM25 rồi trộn bằng reciprocal-rank fusion
($\sum 1/(60 + rank)$), giới hạn số tài liệu mỗi loại nguồn.

**2. Ưu / nhược.** Thêm mô tả hành vi và phản ví dụ lành mà model có thể chưa biết. Nhưng tài liệu giống bề mặt không đồng nghĩa liên quan,
và một ví dụ mã độc gần giống có thể kéo verdict về "độc" cho package lành.

**3. Hợp với dữ liệu nào.** Khi mẫu hành vi lặp lại giữa các package (chiến dịch typosquat dùng cùng payload) và KB có phản ví dụ lành.

In [ ]:
for r in MAIN:
    print('#####', r); table(r, ['e0', 'e1', 'e2', 'e2_bm25', 'e2_rerank']); cmp(r, ['e1_vs_e0', 'e2_vs_e1', 'e2_bm25_vs_e2', 'e2_rerank_vs_e2'])
GOLD = {'gold/kb_recall@1': 'recall@1', 'gold/kb_recall@5': 'recall@5', 'gold/kb_mrr': 'MRR', 'gold/evidence_coverage': 'coverage bằng chứng'}
for r in SYN:
    print('##### (gold, synthetic)', r); table(r, ['e1', 'e2', 'e2_bm25', 'e3', 'e3a'], GOLD)

## Phần 3. KG-RAG hành vi (E3) và đối chứng E3a

**1. Hoạt động thế nào.** Mỗi package thành một đồ thị: `Package -> Module -> Function -> Operation(API)`, hành vi của từng thao tác, và cạnh
`FLOWS_TO` **chỉ khi AST cho thấy giá trị đi từ thao tác này sang thao tác kia** (lồng lời gọi, def-use, giá trị trả về của hàm).
Duyệt 1-3 bước, khớp pattern (ví dụ `env_access -> obfuscation -> network_send`), ưu tiên code chạy lúc cài đặt / import, rồi viết từng đường
thành văn bản có `file:dòng`, extractor và độ tin cậy của mỗi cạnh. E3a đưa **đúng các cạnh đó** dưới dạng câu rời, chọn bằng hybrid retrieval:
nếu E3 hơn E3a thì lợi ích đến từ cấu trúc đường đi, không chỉ từ việc có thêm fact.

**2. Ưu / nhược.** Bằng chứng có quan hệ và trỏ về đúng dòng mã; phân biệt "đọc biến môi trường *và* gửi HTTP trong cùng file" với
"biến môi trường *được gửi* qua HTTP". Nhược: phụ thuộc chất lượng phân tích tĩnh (Python 2, obfuscation, gọi động làm mất cạnh), và một đường
"độc" trong code lành (template engine `compile -> exec`) có thể gây báo nhầm.

**3. Hợp với dữ liệu nào.** Package đầy đủ nhiều file, hành vi độc là *chuỗi* thao tác, không phải một dòng.

In [ ]:
for r in MAIN:
    print('#####', r); table(r, ['e2', 'e3', 'e3a']); cmp(r, ['e3_vs_e2', 'e3_vs_e3a'])
    R = RUNS[r]; s = R['summaries'].get('e3', {})
    print({k: round(v, 3) for k, v in s.items() if k.startswith('graph/') and isinstance(v, float)})
cases = _j(f'{RES}/cases.json') or {}
for r, c in cases.items(): print(r, {k: v[:2] for k, v in c.items()})

## Phần 4. Xác minh bằng chứng (E4)

**1. Hoạt động thế nào.** Mỗi claim trong câu trả lời E3 được kiểm: id trích dẫn có thật trong prompt không, đoạn mã còn khớp hash không, hành vi
có trong fact tĩnh và trong item được trích không. Claim không có chứng cứ bị bỏ; verdict "độc" mà không còn claim nguy hiểm nào được hỗ trợ thì
chuyển thành *uncertain*; verdict "lành" mâu thuẫn với một đường đồ thị nghiêm trọng đã xác nhận bằng AST cũng thành *uncertain*.
Ngưỡng chọn trên validation.

**2. Ưu / nhược.** Giảm lời giải thích bịa mà không cần gọi thêm LLM. Nhưng verifier chỉ kiểm *nhất quán với phân tích tĩnh*: nó có thể loại claim
đúng mà phân tích tĩnh bỏ sót, và abstain làm giảm coverage. Phải đọc F1 strict cùng coverage.

In [ ]:
for r in MAIN:
    print('#####', r); table(r, ['e3', 'e4_cite', 'e4_filter', 'e4']); cmp(r, ['e4_vs_e3', 'e4_filter_vs_e3', 'e4_cite_vs_e3'])

## Dashboard (plan mục 13)
1. F1 theo method, 2. FPR/FNR, 3. claim không chứng cứ theo coverage, 4. recall@K truy hồi (gold, synthetic), 5. chi phí token theo F1, 6. lỗi theo nhóm.

In [ ]:
import matplotlib.pyplot as plt
ORDER = ['e0', 'e1', 'e2', 'e2_bm25', 'e2_rerank', 'e3', 'e3a', 'e4', 'e4_filter', 'e4_cite']
for r in MAIN:
    S = RUNS[r]['summaries']; ms = [m for m in ORDER if m in S]
    fig, ax = plt.subplots(2, 3, figsize=(17, 8)); fig.suptitle(r)
    x = np.arange(len(ms))
    ax[0, 0].bar(x - .2, [S[m]['strict/f1_malicious'] for m in ms], .4, label='F1 độc'); ax[0, 0].bar(x + .2, [S[m]['strict/macro_f1'] for m in ms], .4, label='macro F1')
    lo = [S[m]['strict/f1_malicious'] - S[m]['f1_ci95']['ci_low'] for m in ms]; hi = [S[m]['f1_ci95']['ci_high'] - S[m]['strict/f1_malicious'] for m in ms]
    ax[0, 0].errorbar(x - .2, [S[m]['strict/f1_malicious'] for m in ms], yerr=[lo, hi], fmt='none', ecolor='k', capsize=2)
    ax[0, 1].bar(x - .2, [S[m]['strict/fpr'] for m in ms], .4, label='FPR'); ax[0, 1].bar(x + .2, [S[m]['strict/fnr'] for m in ms], .4, label='FNR')
    for m in ms:
        ax[0, 2].scatter(S[m]['coverage'], S[m]['evidence/unsupported_claim_rate']); ax[0, 2].annotate(m, (S[m]['coverage'], S[m]['evidence/unsupported_claim_rate']))
        ax[1, 1].scatter(S[m]['tokens_in_mean'], S[m]['strict/f1_malicious']); ax[1, 1].annotate(m, (S[m]['tokens_in_mean'], S[m]['strict/f1_malicious']))
    for a, t in ((ax[0, 0], 'F1 (strict, CI 95%)'), (ax[0, 1], 'FPR / FNR')):
        a.set_xticks(x); a.set_xticklabels(ms, rotation=40, ha='right'); a.legend(); a.set_title(t)
    ax[0, 2].set_xlabel('coverage'); ax[0, 2].set_ylabel('claim không chứng cứ'); ax[0, 2].set_title('Hallucination vs coverage')
    ax[1, 1].set_xlabel('token vào / package'); ax[1, 1].set_ylabel('F1 độc'); ax[1, 1].set_title('Chi phí vs F1')
    if SYN:
        Sg = RUNS[SYN[0]]['summaries']
        for m in [m for m in ('e1', 'e2', 'e2_bm25', 'e3a') if m in Sg]:
            ax[1, 0].plot([1, 3, 5], [Sg[m].get(f'gold/kb_recall@{k}', np.nan) for k in (1, 3, 5)], marker='o', label=m)
        ax[1, 0].set_xlabel('K'); ax[1, 0].set_ylabel('recall@K (gold)'); ax[1, 0].legend(); ax[1, 0].set_title('Truy hồi (synthetic)')
    eg = f'{RES}/{r}/error_groups.csv'
    if os.path.exists(eg):
        E = pd.read_csv(eg, index_col=0); E[[m for m in ('e0', 'e2', 'e3', 'e4') if m in E.columns]].plot.bar(ax=ax[1, 2], rot=20)
        ax[1, 2].set_title('Tỉ lệ sai theo nhóm'); ax[1, 2].set_ylabel('tỉ lệ sai')
    fig.tight_layout(); plt.show()

## Kết luận

Tóm tắt tự động bên dưới chỉ nêu kết quả có kiểm định cặp (cùng sample id). Khi viết kết luận:
- nêu rõ **loại dữ liệu** của từng kết luận (`malllm` chỉ có setup.py; `pyguard_obf` là package đầy đủ);
- kết quả âm tính (CI chứa 0) vẫn là kết quả: nó trả lời "khi nào phương pháp *không* đáng chi phí";
- nhãn hallucination là nhãn tự động theo phân tích tĩnh; chỉ trích khi notebook 02 cell 06 cho thấy verifier khớp với người gán nhãn.

In [ ]:
for r in MAIN:
    R = RUNS[r]; C = R['comparisons']; S = R['summaries']
    print('#####', r, '| n =', R['n_samples'])
    for k, txt in [('e1_vs_e0', 'Truy hồi so với LLM thuần'), ('e2_vs_e1', 'Hybrid so với dense'), ('e3_vs_e2', 'KG-RAG so với hybrid'),
                   ('e3_vs_e3a_exact', 'Cấu trúc đường so với đúng các cạnh đó dạng câu rời (E3a)'),
                   ('e3_vs_e3a', 'Duyệt đồ thị so với fact chọn bằng truy hồi (E3b)'), ('e4_vs_e3', 'Xác minh so với không xác minh')]:
        c = C.get(k)
        if not c: continue
        b = c['f1_malicious_strict']
        sig = 'có ý nghĩa' if c['significant_0.05'] else 'không có ý nghĩa thống kê'
        print(f"  {txt}: dF1 = {b['delta']:+.3f} [{b['ci_low']:+.3f}, {b['ci_high']:+.3f}] ({sig}); dFPR = {c['fpr_strict']['delta']:+.3f}")
    if 'e3' in S and 'e4' in S:
        G = R.get('grounding_vs_e3') or {}
        if G.get('e4'):
            print(f"  Claim không chứng cứ / claim gốc (cùng mẫu số): E3 {G['e3']['unsupported_over_original_claims']:.3f} -> "
                  f"E4 {G['e4']['unsupported_over_original_claims']:.3f}; claim giữ lại {G['e4']['claims_retained']:.2f}, coverage E4 {S['e4']['coverage']:.2f}")